# cleaning
lowercase
strip
remove punctuation
remove procedural interventions
tokenize
remove stopwords, don't forget to store raw and cleaned (for debugging reasons)

3 categories I see (for this is necessary to separate the speeches first and then to do a progress of categorization)
* *meta information*
I have to discover everything that its more metainformation on the session (date, session numbers, type of discussion, election results and process).

* *Substantive / political speech*
Also the speeches by the deputies.
They start with the president introducing them, then they start and this is always a new parragraph, also their names appear with a (el/la señor*) start of a parragraph

* *floor control*
There's also the comments made that are about the organization of the chamber itself, comments mainly from the "mesa directiva" to control the floor, talk about rules, closing the speech of people that spoke too long, ask for silence, calling for elections

Pseudocode for exploring tools

Discover "data/extracted_laws" folder
for each of the folder there, select all files that are .txt (if any) and append them to a dictionary (name: text)
Read the whole document
* First line has "Fecha DD de month, YYYY", sesion N°, legislatura N°, type discusión (general, única)
* Second line: title of the law
* Third line: start of the president of the senate (or someone with the power in that moment)
Then the president again talks to give the speaking power to the diputado informante

Every time someone starts talking it's written in the transcript who will talk with this pattern "El señor / La señora(ita) LASTNAME (don FIRSTNAME / doña FIRSTNAME).-"

The first person speaking (the one that appears in the third line) is the person controlling the flow of the discussion (can also be vicepresident), the first speech is to introduce the conversation, second one is to give the voice to the deputy that's "informing" ("el diputado informante / la diputada informante") about the law, who then will give it's description and explanation of the law, sometimes there can be a second informing deputy that will also be introducing by the controlling person.
At some point the controlling speaker will say "en discusión..." from that moment on the discussion is on, then different speakers can start talk (also ministers)

______
Pseudocode for raw_interventions
should get:

    law_id, speaker_raw, dipid(if), name_dip, date, filename/key_section, role_raw, speaker_sex (to compare), speech_raw
____
Pseudocode for pol_speech
should get:

    law_id, date, speaker_raw, name_dip, dipid, role_raw, speech_clean, is_control, is_substantive

In [8]:
# import libraries
from pathlib import Path
import pandas as pd
import json
import re

In [60]:
# set path
LAWS_FOLDER = Path("../../data/extracted_laws")
SAVE_RESULTS = Path("../../data/processed/turns.csv")

In [10]:
def get_laws(root):
    """
    Search recursively for all .json transcript files inside root.
    Read each file and return a dataframe with metadata + transcript text.
    """
    root = Path(root)
    rows = []

    for json_path in root.rglob("*.json"): # rglob searches recursively for all the .json files withing the folder
        with open(json_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        

        # each json file has a law_meta, entry
        law_meta = data.get("law_meta", {})
        entry = data.get("entry", {})
        item_attrs = data.get("item_attrs", {})

        row = {
            # information about the file
            "file_name": json_path.name,
            "json_path": str(json_path),
            
            # information abouot the law itself
            "law_id": law_meta.get("law_id"), # this is how it is stored in the library of the congress
            "law_number": law_meta.get("law_number"), # this is the official number of the law
            #"law_title": law_meta.get("law_title"), # this is wrong, has to be fixed or deleted

            # information about the discussion
            "section_key": entry.get("section_key"),
            "event_date": entry.get("event_date"),

            # transcription of discussion
            "text": data.get("transcript_text"),
        }

        rows.append(row)

    return pd.DataFrame(rows)

In [ ]:
discussions_df = get_laws(LAWS_FOLDER)

In [13]:
discussions_df.info()
discussions_df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28 entries, 0 to 27
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   file_name    28 non-null     object
 1   json_path    28 non-null     object
 2   law_id       28 non-null     object
 3   law_number   28 non-null     object
 4   section_key  28 non-null     object
 5   event_date   28 non-null     object
 6   text         28 non-null     object
dtypes: object(7)
memory usage: 1.7+ KB


In [14]:
def extract_parentheses(discussions_df):
    """
    Extract all text inside parentheses from all transcript texts.
    Return one row per match.
    """
    rows = []
    pattern = re.compile(r"\((.*?)\)")

    for _, row in discussions_df.iterrows():
        text = row["text"]

        if not isinstance(text, str):
            continue

        matches = pattern.findall(text)

        for match in matches:
            rows.append({
                "law_number": row["law_number"],
                "section_key": row["section_key"],
                "event_date": row["event_date"],
                "match": match.strip()
            })

    return pd.DataFrame(rows)

In [15]:
parentheses = extract_parentheses(discussions_df)

In [16]:
parentheses

,law_number,section_key,event_date,match
0,21.472,1-5,2022-06-07,PRIMER TRÁMITE CONSTITUCIONAL. BOLETÍN N° 1499...
1,21.472,1-5,2022-06-07,Presidente
2,21.472,1-5,2022-06-07,Presidente
3,21.472,1-5,2022-06-07,de pie
4,21.472,1-5,2022-06-07,presidenta
...,...,...,...,...
1398,21.800,2-5,2024-12-17,19 del Senado
1399,21.800,2-5,2024-12-17,Vicepresidente
1400,21.800,2-5,2024-12-17,Vicepresidente
1401,21.800,2-5,2024-12-17,Vicepresidente


In [17]:
def show_context(text, phrase, window=2):
    """
    Print lines around a phrase.
    """
    lines = text.splitlines()

    for i, line in enumerate(lines):
        if phrase.lower() in line.lower():
            start = max(0, i - window)
            end = min(len(lines), i + window + 1)

            print("=" * 80)
            for j in range(start, end):
                marker = ">>" if j == i else "  "
                print(f"{marker} {j}: {lines[j]}")

In [ ]:
sample_text = discussions_df.iloc[0]["text"]
show_context(sample_text, "Chile", window=2)

In [ ]:
SPEAKER_HEADER_RE = re.compile(
    r"^(El|La)\s+(señor|señora|señorita)\s+(.+?)\.-\s*$"
)

def collect_speaker_headers(discussions_df):
    """
    Find lines that look like speaker headers.
    Return a dataframe with one row per header line.
    """
    rows = []

    for _, row in discussions_df.iterrows():
        lines = row["text"].splitlines()

        for i, line in enumerate(lines):
            stripped = line.strip()

            if SPEAKER_HEADER_RE.match(stripped):
                rows.append({
                    "law_number": row["law_number"],
                    "section_key": row["section_key"],
                    "file_name": row["file_name"],
                    "line_number": i,
                    "speaker_header_raw": stripped
                })

    return pd.DataFrame(rows)

In [28]:
headers = collect_speaker_headers(discussions_df)
headers

,law_number,section_key,file_name,line_number,speaker_header_raw
0,21.472,1-5,discussion_2022-06-07_1_5.json,4,"El señor SOTO, don Raúl (Presidente).-"
1,21.472,1-5,discussion_2022-06-07_1_5.json,22,"El señor SOTO, don Raúl (Presidente).-"
2,21.472,1-5,discussion_2022-06-07_1_5.json,26,"La señorita AHUMADA, doña Yovana (de pie).-"
3,21.472,1-5,discussion_2022-06-07_1_5.json,118,"El señor SOTO, don Raúl (Presidente).-"
4,21.472,1-5,discussion_2022-06-07_1_5.json,122,El señor NARANJO (de pie).-
...,...,...,...,...,...
1523,21.800,2-5,discussion_2024-12-17_2_5.json,952,El señor AEDO (Vicepresidente).-
1524,21.800,2-5,discussion_2024-12-17_2_5.json,960,El señor AEDO (Vicepresidente).-
1525,21.800,2-5,discussion_2024-12-17_2_5.json,976,El señor AEDO (Vicepresidente).-
1526,21.800,2-5,discussion_2024-12-17_2_5.json,984,El señor AEDO (Vicepresidente).-


In [ ]:
# check the percentage of starting phrases that have a parentheses
total = len(headers)
with_parentheses = headers["speaker_header_raw"].str.contains(r"[()]", regex=True).sum()
without_parentheses = total - with_parentheses

print(f"Total headers: {total}")
print(f"With parentheses: {with_parentheses} ({with_parentheses/total*100:.1f}%)")
print(f"Without parentheses: {without_parentheses} ({without_parentheses/total*100:.1f}%)")


Total headers: 1528
With parentheses: 1214 (79.5%)
Without parentheses: 314 (20.5%)


In [40]:
# check the context (lines surrounding desired text)
def show_context(text, phrase, window=2): #confirm how to create the args
    """
    Print the lines around a phrase.
    window=2 means 2 lines before and 2 lines after.
    """
    lines = text.splitlines()

    for i, line in enumerate(lines):
        if phrase.lower() in line.lower():
            start = max(0, i - window)
            end = min(len(lines), i + window + 1)

            print("=" * 80)
            for j in range(start, end):
                marker = ">>" if j == i else "  "
                print(f"{marker} {j}: {lines[j]}")

In [ ]:
sample_text = discussions_df.iloc[0]["text"] # why iloc?
show_context(sample_text, "Tiene la palabra", window=2)

In [43]:
def split_raw_turns(text):
    """
    Split one transcript into raw speaker turns.
    A new turn starts every time a speaker header appears.
    """
    lines = text.splitlines()
    turns = []

    current_turn = None
    turn_order = 0

    for line in lines:
        stripped = line.strip()

        # skip empty lines
        if stripped == "":
            continue

        header_match = SPEAKER_HEADER_RE.match(stripped)

        # if the line is a speaker header, start a new turn
        if header_match:
            # close the previous turn first
            if current_turn is not None:
                current_turn["speech_raw"] = current_turn["speech_raw"].strip()
                turns.append(current_turn)

            turn_order += 1

            current_turn = {
                "turn_order": turn_order,
                "speaker_header_raw": stripped,
                "speaker_article": header_match.group(1),   # El / La
                "speaker_label": header_match.group(2),     # señor / señora / señorita
                "speaker_body_raw": header_match.group(3),  # the rest of the header
                "speech_raw": ""
            }

        # if it is not a header, add the line to the current speech
        else:
            if current_turn is not None:
                current_turn["speech_raw"] += stripped + "\n"

    # after the loop, save the last turn too
    if current_turn is not None:
        current_turn["speech_raw"] = current_turn["speech_raw"].strip()
        turns.append(current_turn)

    return turns

In [ ]:
turns_disc = split_raw_turns(sample_text)
turns_disc

# trabajar los "\n"

In [47]:
def build_raw_turns_df(laws_df):
    """
    Apply split_raw_turns to every file in laws_df
    and return one dataframe with all turns.
    """
    rows = []

    for _, row in laws_df.iterrows():
        turns = split_raw_turns(row["text"])

        for turn in turns:
            rows.append({
                "law_number": row["law_number"],
                "section_key": row["section_key"],
                "turn_order": turn["turn_order"], #sure?
                "speaker_header_raw": turn["speaker_header_raw"],
                "speaker_article": turn["speaker_article"], # sure?
                "speaker_label": turn["speaker_label"],
                "speaker_body_raw": turn["speaker_body_raw"],
                "speech_raw": turn["speech_raw"]
            })

    return pd.DataFrame(rows)

In [48]:
raw_turns_df = build_raw_turns_df(discussions_df)
raw_turns_df.head(10)

,law_number,section_key,turn_order,speaker_header_raw,speaker_article,speaker_label,speaker_body_raw,speech_raw
0,21.472,1-5,1,"El señor SOTO, don Raúl (Presidente).-",El,señor,"SOTO, don Raúl (Presidente)","Iniciando el Orden del Día, corresponde tratar..."
1,21.472,1-5,2,"El señor SOTO, don Raúl (Presidente).-",El,señor,"SOTO, don Raúl (Presidente)","Doy la bienvenida al ministro de Energía, seño..."
2,21.472,1-5,3,"La señorita AHUMADA, doña Yovana (de pie).-",La,señorita,"AHUMADA, doña Yovana (de pie)","Señor Presidente, en mi calidad de diputada in..."
3,21.472,1-5,4,"El señor SOTO, don Raúl (Presidente).-",El,señor,"SOTO, don Raúl (Presidente)",Tiene la palabra el diputado informante de la ...
4,21.472,1-5,5,El señor NARANJO (de pie).-,El,señor,NARANJO (de pie),"Señor Presidente, honorable Cámara, en nombre ..."
5,21.472,1-5,6,"El señor SOTO, don Raúl (Presidente).-",El,señor,"SOTO, don Raúl (Presidente)",En discusión el proyecto de ley. Tiene la pala...
6,21.472,1-5,7,El señor BRITO .-,El,señor,BRITO,"Señor Presidente, para nosotros es demasiado i..."
7,21.472,1-5,8,"El señor SOTO, don Raúl (Presidente).-",El,señor,"SOTO, don Raúl (Presidente)",Tiene la palabra el diputado Felipe Donoso .
8,21.472,1-5,9,El señor DONOSO .-,El,señor,DONOSO,"Señor Presidente, me gustaría expresar lo defi..."
9,21.472,1-5,10,"El señor SOTO, don Raúl (Presidente).-",El,señor,"SOTO, don Raúl (Presidente)",Tiene la palabra el diputado Francisco Pulgar .


In [49]:
def extract_role(speaker_body_raw):
    """
    Extract the text inside parentheses from the speaker header.
    Example:
    'SEPÚLVEDA (Vicepresidente)' -> 'Vicepresidente'
    """
    match = re.search(r"\((.*?)\)", speaker_body_raw)

    if match:
        return match.group(1).strip()

    return None

In [50]:
raw_turns_df["role_raw"] = raw_turns_df["speaker_body_raw"].apply(extract_role)

In [51]:
raw_turns_df

,law_number,section_key,turn_order,speaker_header_raw,speaker_article,speaker_label,speaker_body_raw,speech_raw,role_raw
0,21.472,1-5,1,"El señor SOTO, don Raúl (Presidente).-",El,señor,"SOTO, don Raúl (Presidente)","Iniciando el Orden del Día, corresponde tratar...",Presidente
1,21.472,1-5,2,"El señor SOTO, don Raúl (Presidente).-",El,señor,"SOTO, don Raúl (Presidente)","Doy la bienvenida al ministro de Energía, seño...",Presidente
2,21.472,1-5,3,"La señorita AHUMADA, doña Yovana (de pie).-",La,señorita,"AHUMADA, doña Yovana (de pie)","Señor Presidente, en mi calidad de diputada in...",de pie
3,21.472,1-5,4,"El señor SOTO, don Raúl (Presidente).-",El,señor,"SOTO, don Raúl (Presidente)",Tiene la palabra el diputado informante de la ...,Presidente
4,21.472,1-5,5,El señor NARANJO (de pie).-,El,señor,NARANJO (de pie),"Señor Presidente, honorable Cámara, en nombre ...",de pie
...,...,...,...,...,...,...,...,...,...
1523,21.800,2-5,82,El señor AEDO (Vicepresidente).-,El,señor,AEDO (Vicepresidente),Corresponde votar en particular el artículo 21...,Vicepresidente
1524,21.800,2-5,83,El señor AEDO (Vicepresidente).-,El,señor,AEDO (Vicepresidente),Aprobado.\n-Votaron por la afirmativa:\nAceved...,Vicepresidente
1525,21.800,2-5,84,El señor AEDO (Vicepresidente).-,El,señor,AEDO (Vicepresidente),Corresponde votar en particular el artículo te...,Vicepresidente
1526,21.800,2-5,85,El señor AEDO (Vicepresidente).-,El,señor,AEDO (Vicepresidente),Aprobado.\n-Votaron por la afirmativa:\nAceved...,Vicepresidente


In [55]:
def classify_turn(role_raw, speech_raw):
    """
    Very rough first classification of a turn.
    This is just version 1.
    """
    role = ""
    speech = ""

    if role_raw is not None:
        role = role_raw.lower()

    if speech_raw is not None:
        speech = speech_raw.lower()

    # people controlling the floor
    if "presidente" in role or "vicepresidente" in role or "vicepresidenta" in role:
        return "floor_control"

    # ministers
    if "ministro" in role or "ministra" in role:
        return "minister"

    # procedural phrases
    if "cerrado el debate" in speech or "en votación" in speech or "pido silencio" in speech:
        return "floor_control"
    
    # law introduction
    if "informante" in speech:
        return "law_introduction"

    # everything else for now
    return "substantive_candidate"

In [56]:
raw_turns_df["turn_type"] = raw_turns_df.apply(
    lambda row: classify_turn(row["role_raw"], row["speech_raw"]),
    axis=1
)

In [61]:
raw_turns_df.to_csv(SAVE_RESULTS, index=False, encoding="utf-8-sig")